# Overfitting and Regularization

A model that memorises its training set but fails on new data is **overfitting**. This notebook:

1. Builds a deliberately overfit model (big network, tiny dataset)
2. Applies each regularization tool one at a time: **Dropout**, **weight decay**, **BatchNorm**, **early stopping**
3. Compares them on the same data, with loss curves and decision boundaries

Everything runs in seconds on CPU.

In [ ]:
import copy
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

## 1. Data: small and noisy on purpose

Two interleaved half-moons with heavy noise. We give the model only **80 training points**, but 520 validation points to measure generalisation honestly.

In [ ]:
X, y = make_moons(n_samples=600, noise=0.35, random_state=SEED)
X = StandardScaler().fit_transform(X)

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

X_train, y_train = X[:80], y[:80]
X_val,   y_val   = X[80:], y[80:]
print(X_train.shape, X_val.shape)

plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train.squeeze(), cmap='coolwarm', edgecolors='k')
plt.title('80 training points'); plt.show()

## 2. Model factory and training function

One factory builds every variant, so each experiment differs by exactly one switch. The training function tracks train and validation loss each epoch and supports early stopping.

In [ ]:
def make_model(hidden=256, dropout=0.0, batchnorm=False):
    layers, in_f = [], 2
    for _ in range(3):
        layers.append(nn.Linear(in_f, hidden))
        if batchnorm:
            layers.append(nn.BatchNorm1d(hidden))
        layers.append(nn.ReLU())
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        in_f = hidden
    layers.append(nn.Linear(hidden, 1))
    return nn.Sequential(*layers)


@torch.no_grad()
def evaluate(model, X, y, criterion):
    model.eval()                       # turns Dropout off, BatchNorm uses running stats
    logits = model(X)
    loss = criterion(logits, y).item()
    acc = ((logits > 0).float() == y).float().mean().item()
    return loss, acc


def fit(model, optimizer, epochs=300, patience=None):
    criterion = nn.BCEWithLogitsLoss()
    hist = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_val, best_state, bad = float('inf'), None, 0

    for epoch in range(epochs):
        model.train()                  # Dropout on, BatchNorm uses batch stats
        optimizer.zero_grad()
        loss = criterion(model(X_train), y_train)
        loss.backward()
        optimizer.step()

        val_loss, val_acc = evaluate(model, X_val, y_val, criterion)
        hist['train_loss'].append(loss.item())
        hist['val_loss'].append(val_loss)
        hist['val_acc'].append(val_acc)

        if patience is not None:
            if val_loss < best_val:
                best_val, best_state, bad = val_loss, copy.deepcopy(model.state_dict()), 0
            else:
                bad += 1
                if bad >= patience:
                    hist['stopped_at'] = epoch
                    break

    if best_state is not None:
        model.load_state_dict(best_state)   # roll back to the best epoch
    return hist

## 3. Baseline: watch it overfit

Big network, no regularization. Training loss keeps falling while validation loss turns around and climbs. That gap is overfitting.

In [ ]:
def run(name, dropout=0.0, batchnorm=False, weight_decay=0.0, patience=None, epochs=300):
    torch.manual_seed(SEED)
    model = make_model(dropout=dropout, batchnorm=batchnorm)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=weight_decay)
    hist = fit(model, opt, epochs=epochs, patience=patience)
    return name, model, hist

results = {}
name, model, hist = run('baseline', weight_decay=0.0)
results[name] = (model, hist)

plt.plot(hist['train_loss'], label='train')
plt.plot(hist['val_loss'], label='val')
plt.xlabel('epoch'); plt.ylabel('BCE loss'); plt.legend(); plt.title('Baseline: overfitting')
plt.show()
print(f"min val loss {min(hist['val_loss']):.3f} at epoch {int(np.argmin(hist['val_loss']))}, "
      f"final val loss {hist['val_loss'][-1]:.3f}, final val acc {hist['val_acc'][-1]:.3f}")

## 4. The four tools

| Tool | What it does | PyTorch |
|---|---|---|
| **Dropout** | Randomly zeroes activations during training, so no neuron can be relied on | `nn.Dropout(p)` |
| **Weight decay** | Penalises large weights, giving smoother functions | `AdamW(..., weight_decay=)` |
| **BatchNorm** | Normalises activations per batch; side effect: mild noise that regularizes | `nn.BatchNorm1d(n)` |
| **Early stopping** | Stop when validation loss stops improving, keep the best weights | loop logic (see `fit`) |

`AdamW` applies weight decay directly to the weights, decoupled from the gradient. With plain `Adam`, `weight_decay` is an L2 term mixed into the gradient and behaves differently. Each step shrinks weights by about `lr × weight_decay`, so with `lr=1e-3` a value of 10 means 1% per step. In this run 0.1 barely helped and 10 worked, so treat it as a knob to tune.

In [ ]:
for name, kw in [
    ('dropout 0.3',       dict(dropout=0.3)),
    ('weight decay 10',   dict(weight_decay=10.0)),
    ('batchnorm',         dict(batchnorm=True)),
    ('early stopping',    dict(patience=20)),
    ('all combined',      dict(dropout=0.3, weight_decay=10.0, batchnorm=True, patience=20)),
]:
    n, m, h = run(name, **kw)
    results[n] = (m, h)

## 5. Comparison

In [ ]:
print(f"{'run':<18}{'best val loss':>14}{'final val loss':>16}{'final val acc':>15}")
for name, (m, h) in results.items():
    # after early stopping the model holds its best weights, so re-evaluate for the true final numbers
    vl, va = evaluate(m, X_val, y_val, nn.BCEWithLogitsLoss())
    print(f"{name:<18}{min(h['val_loss']):>14.3f}{vl:>16.3f}{va:>15.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
for name, (m, h) in results.items():
    ax.plot(h['val_loss'], label=name)
ax.set_ylim(0.2, 1.2); ax.set_xlabel('epoch'); ax.set_ylabel('val loss'); ax.legend()
ax.set_title('Validation loss by regularizer')
plt.show()

## 6. Decision boundaries

A heavily overfit model draws a jagged boundary that wraps individual noisy points. A regularized one draws a smooth curve.

In [ ]:
def boundary(ax, model, title):
    model.eval()
    g = torch.linspace(-3, 3, 200)
    xx, yy = torch.meshgrid(g, g, indexing='xy')
    grid = torch.stack([xx.ravel(), yy.ravel()], dim=1)
    with torch.no_grad():
        z = torch.sigmoid(model(grid)).reshape(xx.shape)
    ax.contourf(xx, yy, z, levels=20, cmap='coolwarm', alpha=0.6)
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train.squeeze(), cmap='coolwarm', edgecolors='k', s=20)
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
boundary(axes[0], results['baseline'][0], 'baseline (overfit)')
boundary(axes[1], results['all combined'][0], 'all combined')
plt.show()

## 7. Gotchas

- **`model.eval()` is mandatory before validation.** With Dropout on, validation predictions are random; with BatchNorm in train mode, they depend on the batch. `evaluate` above sets it every time.
- **Early stopping on the test set is cheating.** Always stop on validation data and report test data only once.
- **Tools are not additive.** Stacking all four (see the table) can end slightly worse than one well-tuned tool, because strong weight decay plus Dropout plus BatchNorm starts to underfit. Add one at a time and measure.
- **Regularize only if you overfit.** If train and val loss are both high, the model is underfitting. Add capacity instead.
- **Data augmentation** is the strongest regularizer for images and costs no parameters: random flips, crops and colour jitter (see `01_basics/04_custom_datasets.ipynb` and `05_cv/02_cifar10_transfer_learning.ipynb`).
- **More data beats every trick here.** Re-run with 400 training points to see the gap shrink on its own.